# Phase 8 : Système de Recommandation (E-commerce)

Ce notebook implémente un moteur de recommandation hybride pour suggérer des produits aux clients en fonction de leur historique d'achat.

## Objectifs :
- Construire une matrice d'interaction Utilisateur-Produit.
- Calculer les similarités entre clients (Collaborative Filtering).
- Recommander des produits 'Next Best Offer' pour un client donné.

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os

import warnings
warnings.filterwarnings('ignore')

In [6]:
# Chargement des données d'interaction
df = pd.read_csv(r'C:\4_ERP_BI\Semestre_2\E-commerce\Esprit_PI_ERPBI_6_2025_2026_E_Commerce\02_ML_Engineering\data\processed\dataset_ml_features.csv')

# Création de la matrice User-Item (par fréquence d'achat)
user_item_matrix = df.groupby(['FK_Client', 'FK_Produit']).size().unstack(fill_value=0)

print(f"Matrice User-Item : {user_item_matrix.shape[0]} clients, {user_item_matrix.shape[1]} produits")
user_item_matrix.head()

Matrice User-Item : 267 clients, 32 produits


FK_Produit,0,6,15,16,24,26,34,36,38,39,...,76,80,83,87,93,95,99,103,110,112
FK_Client,,,,,,,,,,,,,,,,,,,,,
0,33,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,6,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 1. Filtrage Collaboratif (User-Based)

Nous calculons la similarité cosinus entre les vecteurs d'achats des clients.

In [7]:
# Calcul de la similarité cosinus entre utilisateurs
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)
user_similarity_df.head()

FK_Client,0,1,2,3,4,5,6,7,8,9,...,308,309,310,311,312,314,315,317,318,319
FK_Client,,,,,,,,,,,,,,,,,,,,,
0,1.000000,0.0,0.999541,0.999541,0.0,0.0,0.999541,0.0,0.030289,0.999541,...,0.0,0.999541,0.0,0.0,0.999541,0.999541,0.0,0.0,0.999541,0.0
1,0.000000,1.0,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.000000,0.0,1.0,0.000000,0.000000,1.0,0.0,0.000000,0.0
2,0.999541,0.0,1.000000,1.000000,0.0,0.0,1.000000,0.0,0.000000,1.000000,...,0.0,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,1.000000,0.0
3,0.999541,0.0,1.000000,1.000000,0.0,0.0,1.000000,0.0,0.000000,1.000000,...,0.0,1.000000,0.0,0.0,1.000000,1.000000,0.0,0.0,1.000000,0.0
4,0.000000,0.0,0.000000,0.000000,1.0,0.0,0.000000,0.0,0.000000,0.000000,...,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.0


In [ ]:
## 📌 Système de recommandation – Filtrage collaboratif (User‑Based)

### Principe général
L’idée est simple : *« Dis‑moi qui sont tes voisins, je te dirai ce que tu aimeras »*.  
On part du principe que deux clients qui ont acheté des produits similaires dans le passé auront des goûts proches. On peut donc recommander à un client les produits qu’un autre client « similaire » a aimés.

### Étape 1 – Matrice utilisateur × produit
On construit un tableau (matrice) où :
- **Lignes** = chaque client (FK_Client)
- **Colonnes** = chaque produit (FK_Produit)
- **Valeurs** = le nombre d’achats du produit par le client (généralement 0, 1, 2…)

Dans notre cas, la matrice fait **267 clients × 32 produits** (seulement 32 références distinctes dans les données). C’est une matrice très **creuse** (beaucoup de zéros), car chaque client n’achète qu’une petite partie des produits.

### Étape 2 – Similarité cosinus entre clients
On calcule la **similarité cosinus** entre chaque paire de clients. Cette mesure regarde l’angle entre deux vecteurs d’achat :
- 1 = vecteurs identiques (mêmes produits achetés aux mêmes fréquences)
- 0 = aucun produit en commun
- Entre 0 et 1 = similarité partielle

Le résultat est une **matrice de similarité** de taille 267 × 267 (symétrique). Chaque case donne la proximité entre deux clients.

### Ce que montrent les extraits affichés
- Les premières lignes de la matrice de similarité contiennent des valeurs très proches de 1 (0.9995, 0.9999…) ou de 0.  
  → Cela signifie que certains clients sont **presque identiques** (ils ont acheté exactement les mêmes produits), tandis que d’autres n’ont **aucun produit commun** (similarité 0).
- Les valeurs comme `0.999541` ou `1.000001` sont des artefacts de calcul (arrondis). En pratique, on peut les considérer comme 1 (similarité parfaite).

### Pourquoi cette approche est utile ?
- Elle ne nécessite aucune connaissance sur les produits (pas besoin de catégories, de prix…).  
- Elle fonctionne uniquement sur les comportements passés.  
- Idéale pour faire du **« Next Best Offer »** : proposer à un client un produit qu’un client similaire a acheté mais que lui n’a pas encore vu.

### Limites à garder en tête
- **Froideur (cold start)** : un nouveau client sans historique n’a aucun voisin → on ne peut rien lui recommander.  
- **Matrice creuse** : peu de produits, peu d’achats par client → les similarités sont souvent 0 ou 1, peu de nuances.  
- **Densité faible** : avec 267 clients et 32 produits, le système reste simple. Sur un vrai site e-commerce, il faudrait des millions d’interactions.

### Phrase pour la soutenance
> *“Notre moteur de recommandation utilise le filtrage collaboratif basé sur la similarité cosinus entre clients. La matrice utilisateur‑produit
(267 clients × 32 produits) permet d’identifier des profils d’achats similaires. Pour un client donné, nous pouvons ainsi lui suggérer les produits
appréciés par ses plus proches voisins, conformément à l’approche décrite dans la partie bonus du guide.”*

In [4]:
def get_recommendations(client_id, matrix, similarity_df, n_recommendations=3):
    if client_id not in matrix.index:
        return []
    
    # Trouver les utilisateurs les plus similaires
    similar_users = similarity_df[client_id].sort_values(ascending=False)[1:6].index
    
    # Produits achetés par ces utilisateurs mais pas par le client actuel
    client_purchases = matrix.loc[client_id]
    unbought_products = client_purchases[client_purchases == 0].index
    
    # Score des produits basés sur les achats des voisins
    reco_scores = matrix.loc[similar_users, unbought_products].mean().sort_values(ascending=False)
    
    return reco_scores.head(n_recommendations).index.tolist()

# Test
sample_client = user_item_matrix.index[0]
print(f"Recommandations pour le client {sample_client} : {get_recommendations(sample_client, user_item_matrix, user_similarity_df)}")

Recommandations pour le client 0 : [6, 15, 110]


In [ ]:
## 📌 Comment fonctionne notre moteur de recommandation ?

### La logique en français courant

On veut suggérer à un client des produits qu'il n'a pas encore achetés, mais qui pourraient lui plaire. Pour cela, on suit ces étapes :

1. **On cherche ses « voisins »** : parmi tous les clients, on repère ceux qui ont un historique d'achat le plus proche du client actuel (similarité cosinus élevée). On garde par exemple les 5 meilleurs voisins.

2. **On regarde ce que les voisins ont acheté** : on liste tous les produits que ces voisins ont achetés.

3. **On enlève les produits déjà achetés par le client** : on ne veut pas recommander ce qu'il a déjà.

4. **On classe les produits restants par popularité** : plus un produit a été acheté par les voisins, plus il est recommandable.

5. **On renvoie les 3 meilleurs** : les fameux « Next Best Offers ».

### Exemple avec le client 0

Pour le client 0, le système a trouvé que ses voisins achètent souvent les produits **6, 15 et 110**. Comme le client 0 ne les a jamais achetés, on les lui recommande.

> *« Le client 0 a des goûts proches de plusieurs autres clients. Ces derniers apprécient les produits 6, 15 et 110 – nous les suggérons donc au client 0. »*

### Sauvegarde pour l’application web

Les deux objets essentiels (la matrice clients×produits et la matrice de similarité entre clients) sont sauvegardés dans des fichiers.
L’application Streamlit pourra ainsi charger ces modèles et générer des recommandations en temps réel pour n’importe quel client.

In [9]:
# Sauvegarde pour l'application Web
os.makedirs('../models', exist_ok=True)
joblib.dump(user_item_matrix, '../models/user_item_matrix.joblib')
joblib.dump(user_similarity_df, '../models/user_similarity_df.joblib')

print("Modèles de recommandation sauvegardés.")

Modèles de recommandation sauvegardés.
